
## Collaborative Filtering with ALS

### 1. Objective

The objective of this notebook is to build a personalised recommendation model using customers' historical purchase interactions.

An implicit-feedback Alternating Least Squares (ALS) model will be trained using customer–article interactions available before the evaluation period.

The model will generate up to 12 recommendations per customer and will be evaluated against purchases made during the following 7-day period using Recall@12 and MAP@12.

Performance will be compared against the strongest popularity baseline established previously.

In [3]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

### 1. Load Data and Recreate the Temporal Split

The same temporal evaluation strategy used for the popularity baselines is
retained.

The final seven days of transactions represent future customer behaviour,
while all earlier transactions are available to the recommender.

In [4]:
transactions = pd.read_csv(
    "../data/raw/transactions_train.csv",
    usecols=["t_dat", "customer_id", "article_id"]
)

transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

max_date = transactions["t_dat"].max()
eval_start = max_date - pd.Timedelta(days=6)

train = transactions[
    transactions["t_dat"] < eval_start
].copy()

evaluation = transactions[
    transactions["t_dat"] >= eval_start
].copy()

print("Train:", train["t_dat"].min(), "to", train["t_dat"].max())
print("Evaluation:", evaluation["t_dat"].min(), "to", evaluation["t_dat"].max())

Train: 2018-09-20 00:00:00 to 2020-09-15 00:00:00
Evaluation: 2020-09-16 00:00:00 to 2020-09-22 00:00:00


### 2. Prepare Customer–Article Interactions

ALS operates on a user–item interaction matrix rather than individual
transaction rows.

Multiple purchases of the same article by the same customer are therefore
aggregated into a purchase count.

In [5]:
interactions = (
    train
    .groupby(["customer_id", "article_id"])
    .size()
    .reset_index(name="purchase_count")
)

interactions.head()

,customer_id,article_id,purchase_count
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,176209023,1
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601006,2
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601043,1
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,607642008,1
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,625548001,1


In [6]:
interactions["purchase_count"].describe()

count    2.710115e+07
mean     1.164084e+00
std      5.722045e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      5.700000e+02
Name: purchase_count, dtype: float64

### 4. Encode customers and articles
ALS works with integer matrix positions rather than the original identifiers.

In [7]:
customer_ids = interactions["customer_id"].unique()
article_ids = interactions["article_id"].unique()

In [8]:
customer_to_idx = {
    customer_id: idx
    for idx, customer_id in enumerate(customer_ids)
}

article_to_idx = {
    article_id: idx
    for idx, article_id in enumerate(article_ids)
}

In [10]:
idx_to_article = {
    idx: article_id
    for article_id, idx in article_to_idx.items()
}
interactions["user_idx"] = interactions["customer_id"].map(customer_to_idx)
interactions["item_idx"] = interactions["article_id"].map(article_to_idx)

In [11]:
interactions[
    ["customer_id", "article_id", "purchase_count", "user_idx", "item_idx"]
].head()

,customer_id,article_id,purchase_count,user_idx,item_idx
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,176209023,1,0,0
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601006,2,0,1
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601043,1,0,2
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,607642008,1,0,3
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,625548001,1,0,4


### 5. Build the sparse interaction matrix
The theoretical customer × article matrix is enormous, but almost all entries are empty.

A sparse matrix stores only observed interactions.

In [14]:
binary_values = np.ones(
    len(interactions),
    dtype=np.float32
)
binary_matrix = csr_matrix(
    (
        binary_values,
        (
            interactions["user_idx"],
            interactions["item_idx"]
        )
    ),
    shape=(
        len(customer_ids),
        len(article_ids)
    )
)

In [15]:
binary_matrix.shape

(1356709, 103880)

In [16]:
binary_matrix.nnz

27101148